### GIL

- GIL：GIL (Global Interpreter Lock，全局解释器锁) 是 CPython 解释器层面的一把互斥锁（Mutex）。其核心使命是保护 Python 内部的对象模型（尤其是基于引用计数 Reference Counting 的内存管理机制）免受多线程并发修改造成的竞态条件（Race Condition）破坏。
    - GIL 的硬性限制：无论计算机拥有多少个 CPU 核心，在任何一个微观的瞬间，只有一个操作系统线程能够获取 GIL 并执行 Python 字节码。

### Python 多线程、多进程与协程

- references
    - https://gemini.google.com/app/605701b2f009ee02

- Python 多线程
    - 多线程 (Multi-threading)：内核级的“抢占式”博弈，Python 的 threading 模块调用的是操作系统原生线程（OS-level Threads，如 POSIX Threads）。
    - 调度机制（抢占式调度 Preemptive Scheduling）：线程的挂起与恢复由操作系统的内核调度器决定。不管线程当下的代码有没有执行完，时间片一到，OS 就会强制进行上下文切换（Context Switch）。
    - GIL 视角下的困局：多个线程在内核层面是并行的，但一进入 CPython 解释器，就必须排队争抢 GIL。这导致在 CPU 密集型任务中，多线程不仅无法利用多核，反而会因为极其频繁的锁竞争和线程切换，导致执行效率低于单线程（即所谓的 Thrashing 现象）。
- Python multiprocessing（多进程）模块通过创建全新的操作系统进程来运行您的代码。其“真正并行”的核心原因在于：
    - 独立的解释器实例：每个新进程都有自己独立的 Python 解释器（CPython）和独立的内存空间。
    - 独立的 GIL：因为每个进程都有自己的解释器，所以它们各自拥有一个独立的全局解释器锁（GIL）。
    - 多核利用：操作系统可以将这些独立的进程分配到计算机的不同 CPU 核心上，使它们能够在同一微观时刻真正地同时执行计算。
    - 总结：如果你需要利用多核 CPU 处理计算密集型任务（如图像处理、数学计算），多进程是 Python 中的首选方案。
- Python 协程 (Coroutines)：用户态的“协作式”统筹
    - Python 的 asyncio 则是纯粹在用户态（User Space）通过软件逻辑构建的轻量级并发模型。
    - 调度机制（协作式调度 Cooperative Scheduling）：协程依靠事件循环（Event Loop）来分配时间。执行权的交接是显式且主动的（通常发生在使用 await 遇到 I/O 阻塞时）。
    - GIL 视角下的降维打击：协程本质上运行在单线程内。因为它只有一个 OS 线程，所以从始至终就不存在争抢 GIL 的问题（默认持有）。协程的智慧在于榨干这个单线程在等待 I/O（如网络请求、磁盘读写）时的碎片时间。
- 并发与并行
    - 并发是处理多个任务的结构设计（多线程与协程都是并发机制）。
    - 并行是同时执行多个任务的物理能力。在 Python 中，受 GIL 约束，多线程通常只能做到并发，无法做到真正的 CPU 指令并行（除非绕过 GIL）。
        - Python 多进程是真正的并行

| 特性 | Python 多线程 (`threading`) | Python 协程 (`asyncio`) |
| :--- | :--- | :--- |
| 底层主体 | 操作系统原生线程 (OS Threads) | 用户态轻量级任务 (Tasks/Coroutines) |
| 调度方式 | **抢占式 (Preemptive)**：由操作系统内核强制切换。线程随时可能被暂停。 | **协作式 (Cooperative)**：由用户代码（通过 `await`）主动交出控制权。 |
| GIL 影响 | 多个线程争抢唯一一个 GIL。任一时刻只有一个线程能执行代码。 | 运行在单线程内。不存在锁竞争，默认持有 GIL，在等待 I/O 时主动释放。 |
| 开销 | 线程切换需陷入内核态，开销较高（微秒级）。内存占用较大（M级）。 | 协程切换仅在用户态修改指针，开销极低（纳秒级）。内存占用极小（K级）。 |
| 最适用场景 | 传统的、阻塞式的 I/O 密集型任务，或者需要调用 C 扩展的任务。 | 高并发、非阻塞的 I/O 密集型任务（如网络爬虫、异步 Web 服务器）。 |

### 多线程 vs. 协程

- 下面的两个示例分别模拟了两个任务并发执行，且每个任务都会遇到“等待”（I/O）的情况。
- 多线程的抢占式调度
    - 在这个例子中，即使我们将 num_shared 初始化为 0，你也会发现由于操作系统的抢占，导致了不可预测的结果（竞争条件），因此必须引入额外的锁 (Lock) 来同步。
- 协程始终运行在单线程内，我们显式地使用 await 在“休息”时交出控制权。由于同一时刻只有一个协程在修改变量，因此不需要锁。

In [9]:
import threading
import sys
import time

# 施加极端的 1 微秒切换时间限制
sys.setswitchinterval(1e-6)

counter_real_crash = 0
loop_count = 1_000_000

# 构造一个极简的空函数
# 它的唯一使命：触发底层 CALL 字节码，迫使解释器在此刻检查并释放 GIL
def gil_breaker():
    pass

def task_crash():
    global counter_real_crash
    for _ in range(loop_count):
        # 步骤 A：读取当前值
        temp = counter_real_crash
        
        # 致命一击：强制触发 GIL 重新评估，操作系统在此刻残酷地切走线程
        gil_breaker() 
        
        # 步骤 B：执行写回。此时 temp 里的值极大概率已经过期，导致“丢失的更新”
        counter_real_crash = temp + 1

# 启动 5 个毫无防护的线程
threads = [threading.Thread(target=task_crash) for _ in range(5)]

start = time.time()
for t in threads: t.start()
for t in threads: t.join()
end = time.time()

print("=== 真实的无锁多线程灾难 (Python 3.10+ 被击穿) ===")
print(f"理论期望值: {5 * loop_count}")
print(f"实际输出值: {counter_real_crash} (这一次，必定发生严重的数据崩塌)")
print(f"执行耗时:   {end - start:.4f} 秒\n")

=== 真实的无锁多线程灾难 (Python 3.10+ 被击穿) ===
理论期望值: 5000000
实际输出值: 1755682 (这一次，必定发生严重的数据崩塌)
执行耗时:   0.7385 秒



In [10]:
counter_locked = 0
lock = threading.Lock()

def task_with_lock():
    global counter_locked
    for _ in range(loop_count):
        # 线程获取锁。如果锁被别人占用，该线程立刻陷入阻塞（内核级等待）
        with lock:
            counter_locked += 1

threads_protected = [threading.Thread(target=task_with_lock) for _ in range(5)]

start = time.time()
for t in threads_protected: t.start()
for t in threads_protected: t.join()
end = time.time()

print("=== 2. 加锁多线程 (强制同步) ===")
print(f"理论期望值: {5 * loop_count}")
print(f"实际输出值: {counter_locked} (绝对精准)")
print(f"执行耗时:   {end - start:.4f} 秒 (因高频锁竞争，耗时剧增)\n")

=== 2. 加锁多线程 (强制同步) ===
理论期望值: 5000000
实际输出值: 5000000 (绝对精准)
执行耗时:   1.9678 秒 (因高频锁竞争，耗时剧增)



```python
import asyncio
import time

loop_count = 1_000_000
counter_async = 0

async def task_coroutine():
    global counter_async
    for _ in range(loop_count):
        # 协程环境下，运行在绝对的物理单线程内，无需任何锁！
        counter_async += 1
        
    # 只有显式声明 await 时，协程才会礼貌且主动地交出执行权
    await asyncio.sleep(0) 

async def main():
    # 并发运行 5 个协程任务
    await asyncio.gather(*(task_coroutine() for _ in range(5)))

start = time.time()
asyncio.run(main())
end = time.time()

print("=== 3. 协程模型 (协作式单线程) ===")
print(f"理论期望值: {5 * loop_count}")
print(f"实际输出值: {counter_async} (绝对精准)")
print(f"执行耗时:   {end - start:.4f} 秒 (极速，彻底剥离了内核态切换开销)")
```

- await asyncio.sleep(0) 的真实语义是：“我现在虽然没有 I/O 阻塞，但我自愿放弃当前的时间片，退回到队列的最后面，让别人先跑一次。”

### threading：生产者消费者

> scripts/producer-consumer.py

- 实现生产者-消费者模型 (Producer-Consumer Model)，并结合了守护线程 (Daemon Thread) 与阻塞队列 (Blocking Queue) 技术。
    - q.put(url)：生产；
    - queue.Queue：线程安全；
    - t.daemon = True 将工作线程设为了“守护线程”
        - 一旦主程序退出，这些线程会被强制清退，哪怕任务还在进行中。